# Sales Performance & Profitability Analytics

## 1. Project Overview
This project presents an end-to-end data analytics study of global sales performance, cost structures, and profitability margins across multiple geographic regions, product categories, sales channels, and sales representatives.

### Core Analytical Objectives:
- **Data Integrity & Preprocessing**: Clean raw transactional data, format monetary figures and percentages, parse temporal dimensions.
- **Revenue & Profitability Assessment**: Analyze gross revenue, operating costs, net profit, and profit margins.
- **Geographic & Category Segmentation**: Identify high-performing regions and merchandise categories.
- **Channel & Sales Rep Performance**: Evaluate sales channel contributions and sales force efficiency.
- **Visual Insights & Reporting**: Generate clear visualizations and extract data-grounded business intelligence.

## 2. Import Libraries
Import essential Python libraries for data manipulation, analysis, and visualization.

In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set global visualization style
sns.set_theme(style="whitegrid")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["axes.edgecolor"] = "#cccccc"
plt.rcParams["axes.linewidth"] = 1.0

## 3. Load Dataset
Load the dataset using a portable relative path that resolves seamlessly regardless of working directory.

In [4]:
data_path = "data/44.csv" if os.path.exists("data/44.csv") else "../data/44.csv"
fig_dir = "outputs/figures" if os.path.exists("outputs") else "../outputs/figures"
reports_dir = "outputs/reports" if os.path.exists("outputs") else "../outputs/reports"

os.makedirs(fig_dir, exist_ok=True)
os.makedirs(reports_dir, exist_ok=True)

df = pd.read_csv(data_path)
print(f"Dataset successfully loaded from: {data_path}")
df.head()

Dataset successfully loaded from: ../data/44.csv


## 4. Data Overview
Inspect the schema, dimensions, data types, and initial records of the raw dataset.

In [6]:
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns\n")
print("--- Column Data Types ---")
print(df.dtypes)

print("\n--- First 5 Records ---")
display(df.head())

print("\n--- Raw Dataset Summary ---")
display(df.describe(include="all"))

Dataset Dimensions: 1000 rows, 14 columns

--- Column Data Types ---
Order Date    object
Region        object
Country       object
Category      object
Product       object
Sales Rep     object
Channel       object
Units Sold     int64
Unit Price    object
Unit Cost     object
Revenue       object
Cost          object
Profit        object
Margin %      object
dtype: object

--- First 5 Records ---
   Order Date Region  Country  ...       Cost     Profit Margin %
0  2025-01-01   West  Germany  ...  $1,231.94  $1,092.75    47.0%
1  2025-01-01  North      USA  ...  $2,235.60  $2,624.13    54.0%
2  2025-01-02  North   Canada  ...    $268.64    $211.20    44.0%
3  2025-01-02  North      USA  ...    $539.90    $360.00    40.0%
4  2025-01-02   East    Japan  ...    $296.45    $308.44    51.0%

[5 rows x 14 columns]

--- Raw Dataset Summary ---
        Order Date Region Country Category  ...  Revenue       Cost  Profit  Margin %
count         1000   1000    1000     1000  ...     1000       1

## 5. Data Cleaning & Preprocessing
Clean non-numeric string formatting (currency symbols `$` and commas `,`, percentage signs `%`), parse date fields into `datetime` objects, and extract temporal features (`Year`, `Month`, `Month Name`, `Day`, `Day Name`). Check for missing and duplicate values.

In [8]:
# Missing and Duplicate Checks
print("Missing Values per Column:")
print(df.isnull().sum())
print(f"\nDuplicate Rows: {df.duplicated().sum()}")
df = df.drop_duplicates()

# Clean Currency Columns
currency_columns = ["Unit Price", "Unit Cost", "Revenue", "Cost", "Profit"]
for col in currency_columns:
    df[col] = df[col].astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False).astype(float)

# Clean Percentage Column
df["Margin %"] = df["Margin %"].astype(str).str.replace("%", "", regex=False).astype(float)

# Temporal Feature Extraction
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["Month Name"] = df["Order Date"].dt.month_name()
df["Day"] = df["Order Date"].dt.day
df["Day Name"] = df["Order Date"].dt.day_name()

print("\n--- Cleaned Column Types ---")
print(df.dtypes)
print(f"\nDate Range: {df['Order Date'].min().strftime('%Y-%m-%d')} to {df['Order Date'].max().strftime('%Y-%m-%d')}")

Missing Values per Column:
Order Date    0
Region        0
Country       0
Category      0
Product       0
Sales Rep     0
Channel       0
Units Sold    0
Unit Price    0
Unit Cost     0
Revenue       0
Cost          0
Profit        0
Margin %      0
dtype: int64

Duplicate Rows: 0

--- Cleaned Column Types ---
Order Date    datetime64[ns]
Region                object
Country               object
Category              object
Product               object
Sales Rep             object
Channel               object
Units Sold             int64
Unit Price           float64
Unit Cost            float64
Revenue              float64
Cost                 float64
Profit               float64
Margin %             float64
Year                   int32
Month                  int32
Month Name            object
Day                    int32
Day Name              object
dtype: object

Date Range: 2025-01-01 to 2025-12-31


## 6. Exploratory Data Analysis
Review statistical distribution metrics for cleaned numeric variables.

In [10]:
numeric_cols = ["Units Sold", "Unit Price", "Unit Cost", "Revenue", "Cost", "Profit", "Margin %"]
display(df[numeric_cols].describe().round(2))

       Units Sold  Unit Price  Unit Cost  Revenue     Cost   Profit  Margin %
count     1000.00     1000.00    1000.00  1000.00  1000.00  1000.00   1000.00
mean        21.36       72.72      36.90  1539.75   783.89   755.86     47.20
std         11.60       45.99      20.65  1365.13   647.16   731.30      5.53
min          1.00       14.99       8.69    14.99     8.69     6.30     37.00
25%         10.00       34.99      19.24   539.82   286.74   236.25     42.00
50%         22.00       59.99      34.79  1119.72   595.14   516.38     47.00
75%         31.00       99.99      53.99  2159.64  1113.65  1017.44     53.00
max         40.00      179.99      82.80  7199.60  3312.00  3887.60     54.00


## 7. Sales Analysis
Analyze physical sales volumes across sales dimensions.

In [12]:
total_units = df["Units Sold"].sum()
avg_units = df["Units Sold"].mean()
print(f"Total Units Sold: {total_units:,}")
print(f"Average Units per Order: {avg_units:.2f}")

units_by_region = df.groupby("Region")["Units Sold"].sum().sort_values(ascending=False)
print("\n--- Units Sold by Region ---")
print(units_by_region)

Total Units Sold: 21,355
Average Units per Order: 21.36

--- Units Sold by Region ---
Region
North    5676
South    5451
East     5325
West     4903
Name: Units Sold, dtype: int64


## 8. Revenue Analysis
Evaluate revenue generation across regions, product categories, and sales channels.

In [14]:
total_revenue = df["Revenue"].sum()
avg_revenue = df["Revenue"].mean()
print(f"Total Gross Revenue: ${total_revenue:,.2f}")
print(f"Average Revenue per Order: ${avg_revenue:,.2f}")

revenue_by_region = df.groupby("Region")["Revenue"].sum().sort_values(ascending=False)
revenue_by_category = df.groupby("Category")["Revenue"].sum().sort_values(ascending=False)
revenue_by_channel = df.groupby("Channel")["Revenue"].sum().sort_values(ascending=False)

print("\n--- Revenue by Region ---")
print(revenue_by_region.map("${:,.2f}".format))
print("\n--- Revenue by Category ---")
print(revenue_by_category.map("${:,.2f}".format))
print("\n--- Revenue by Channel ---")
print(revenue_by_channel.map("${:,.2f}".format))

Total Gross Revenue: $1,539,751.45
Average Revenue per Order: $1,539.75

--- Revenue by Region ---
Region
South    $434,395.49
North    $394,483.24
East     $359,386.75
West     $351,485.97
Name: Revenue, dtype: object

--- Revenue by Category ---
Category
Home & Kitchen    $454,309.59
Office            $402,168.54
Apparel           $360,186.15
Electronics       $323,087.17
Name: Revenue, dtype: object

--- Revenue by Channel ---
Channel
Online          $802,328.49
Retail Store    $482,622.42
Wholesale       $254,800.54
Name: Revenue, dtype: object


## 9. Cost & Profit Analysis
Examine total operating cost, net profit, and profit margins.

In [16]:
total_cost = df["Cost"].sum()
total_profit = df["Profit"].sum()
overall_margin = (total_profit / total_revenue) * 100

print(f"Total Cost: ${total_cost:,.2f}")
print(f"Total Net Profit: ${total_profit:,.2f}")
print(f"Overall Profit Margin: {overall_margin:.2f}%")
print(f"Minimum Order Profit: ${df['Profit'].min():,.2f}")
print(f"Maximum Order Profit: ${df['Profit'].max():,.2f}")

profit_by_region = df.groupby("Region")["Profit"].sum().sort_values(ascending=False)
profit_by_category = df.groupby("Category")["Profit"].sum().sort_values(ascending=False)

print("\n--- Profit by Region ---")
print(profit_by_region.map("${:,.2f}".format))
print("\n--- Profit by Category ---")
print(profit_by_category.map("${:,.2f}".format))

Total Cost: $783,890.97
Total Net Profit: $755,860.48
Overall Profit Margin: 49.09%
Minimum Order Profit: $6.30
Maximum Order Profit: $3,887.60

--- Profit by Region ---
Region
South    $218,807.00
North    $191,773.03
East     $172,703.46
West     $172,576.99
Name: Profit, dtype: object

--- Profit by Category ---
Category
Office            $207,714.30
Home & Kitchen    $205,164.22
Apparel           $181,110.53
Electronics       $161,871.43
Name: Profit, dtype: object


## 10. Regional Analysis
Deep dive into regional sales performance, order counts, revenue, profit, and margin percentages.

In [18]:
regional_summary = df.groupby("Region").agg(
    Order_Count=("Order Date", "count"),
    Units_Sold=("Units Sold", "sum"),
    Total_Revenue=("Revenue", "sum"),
    Total_Cost=("Cost", "sum"),
    Total_Profit=("Profit", "sum")
)
regional_summary["Profit_Margin_%"] = (regional_summary["Total_Profit"] / regional_summary["Total_Revenue"]) * 100
display(regional_summary.sort_values(by="Total_Revenue", ascending=False).round(2))

        Order_Count  Units_Sold  ...  Total_Profit  Profit_Margin_%
Region                           ...                               
South           251        5451  ...     218807.00            50.37
North           265        5676  ...     191773.03            48.61
East            253        5325  ...     172703.46            48.06
West            231        4903  ...     172576.99            49.10

[4 rows x 6 columns]


## 11. Product & Category Analysis
Identify individual product performance leaders and aggregate category contributions.

In [20]:
product_summary = df.groupby("Product").agg(
    Category=("Category", "first"),
    Units_Sold=("Units Sold", "sum"),
    Total_Revenue=("Revenue", "sum"),
    Total_Profit=("Profit", "sum")
)
product_summary["Profit_Margin_%"] = (product_summary["Total_Profit"] / product_summary["Total_Revenue"]) * 100

print("--- Top 5 Most Profitable Products ---")
display(product_summary.sort_values(by="Total_Profit", ascending=False).head(5).round(2))

print("\n--- Bottom 5 Least Profitable Products ---")
display(product_summary.sort_values(by="Total_Profit", ascending=True).head(5).round(2))

--- Top 5 Most Profitable Products ---
                     Category  Units_Sold  ...  Total_Profit  Profit_Margin_%
Product                                    ...                               
Desk Chair             Office        1440  ...     139953.60            54.00
Cookware Set   Home & Kitchen        1553  ...      98755.27            53.00
Smart Watch       Electronics        1119  ...      90627.81            54.00
Jacket                Apparel        1341  ...      72400.59            54.00
Running Shoes         Apparel        1372  ...      48363.00            47.01

[5 rows x 5 columns]

--- Bottom 5 Least Profitable Products ---
                    Category  Units_Sold  ...  Total_Profit  Profit_Margin_%
Product                                   ...                               
Notebook Set          Office        1607  ...      10124.10            42.03
Laptop Stand     Electronics        1133  ...      14434.42            50.98
Sunglasses           Apparel        1186 

## 12. Sales Channel Analysis
Evaluate revenue share and profit performance across Online, Retail Store, and Wholesale channels.

In [22]:
channel_summary = df.groupby("Channel").agg(
    Order_Count=("Order Date", "count"),
    Total_Revenue=("Revenue", "sum"),
    Total_Profit=("Profit", "sum")
)
channel_summary["Revenue_Share_%"] = (channel_summary["Total_Revenue"] / total_revenue) * 100
channel_summary["Profit_Margin_%"] = (channel_summary["Total_Profit"] / channel_summary["Total_Revenue"]) * 100
display(channel_summary.sort_values(by="Total_Revenue", ascending=False).round(2))

              Order_Count  Total_Revenue  ...  Revenue_Share_%  Profit_Margin_%
Channel                                   ...                                  
Online                540      802328.49  ...            52.11            48.92
Retail Store          299      482622.42  ...            31.34            49.14
Wholesale             161      254800.54  ...            16.55            49.52

[3 rows x 5 columns]


## 13. Sales Representative Analysis
Benchmark sales rep performance by revenue, profit generated, and order volume.

In [24]:
sales_rep_summary = df.groupby("Sales Rep").agg(
    Order_Count=("Order Date", "count"),
    Units_Sold=("Units Sold", "sum"),
    Total_Revenue=("Revenue", "sum"),
    Total_Profit=("Profit", "sum")
)
sales_rep_summary["Profit_Margin_%"] = (sales_rep_summary["Total_Profit"] / sales_rep_summary["Total_Revenue"]) * 100
display(sales_rep_summary.sort_values(by="Total_Revenue", ascending=False).round(2))

            Order_Count  Units_Sold  ...  Total_Profit  Profit_Margin_%
Sales Rep                            ...                               
K. Ivanova          140        3297  ...     119848.63            49.29
T. Novak            116        2516  ...     101015.71            49.39
S. Owusu            128        2820  ...      97539.59            49.64
A. Patel            128        2636  ...      96668.50            49.27
M. Silva            126        2505  ...      90884.88            49.34
L. Rossi            120        2500  ...      88044.51            48.71
J. Kim              130        2751  ...      87076.68            49.06
R. Chen             112        2330  ...      74781.98            47.65

[8 rows x 5 columns]


## 14. Visualizations
Generate, format, and save all key analytical charts to the `outputs/figures/` directory.

In [26]:
# 1. Revenue by Region
plt.figure(figsize=(8, 5))
bars = plt.bar(revenue_by_region.index, revenue_by_region.values, color="#1f77b4")
plt.title("Revenue by Region", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Region", fontsize=11)
plt.ylabel("Revenue ($)", fontsize=11)
plt.gca().yaxis.set_major_formatter('${x:,.0f}')
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 5000, f'${height:,.0f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "revenue_by_region.png"), dpi=300)
plt.show()

# 2. Profit by Region
plt.figure(figsize=(8, 5))
bars = plt.bar(profit_by_region.index, profit_by_region.values, color="#2ca02c")
plt.title("Profit by Region", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Region", fontsize=11)
plt.ylabel("Profit ($)", fontsize=11)
plt.gca().yaxis.set_major_formatter('${x:,.0f}')
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 3000, f'${height:,.0f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "profit_by_region.png"), dpi=300)
plt.show()

# 3. Revenue by Category
plt.figure(figsize=(8, 5))
bars = plt.bar(revenue_by_category.index, revenue_by_category.values, color="#ff7f0e")
plt.title("Revenue by Category", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Category", fontsize=11)
plt.ylabel("Revenue ($)", fontsize=11)
plt.xticks(rotation=15)
plt.gca().yaxis.set_major_formatter('${x:,.0f}')
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 5000, f'${height:,.0f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "revenue_by_category.png"), dpi=300)
plt.show()

# 4. Profit by Category
plt.figure(figsize=(8, 5))
bars = plt.bar(profit_by_category.index, profit_by_category.values, color="#9467bd")
plt.title("Profit by Category", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Category", fontsize=11)
plt.ylabel("Profit ($)", fontsize=11)
plt.xticks(rotation=15)
plt.gca().yaxis.set_major_formatter('${x:,.0f}')
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 3000, f'${height:,.0f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "profit_by_category.png"), dpi=300)
plt.show()

# 5. Profit by Product
product_profit = df.groupby("Product")["Profit"].sum().sort_values()
plt.figure(figsize=(10, 7))
bars = plt.barh(product_profit.index, product_profit.values, color="#17becf")
plt.title("Profit by Product", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Profit ($)", fontsize=11)
plt.ylabel("Product", fontsize=11)
plt.gca().xaxis.set_major_formatter('${x:,.0f}')
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "profit_by_product.png"), dpi=300)
plt.show()

# 6. Daily Revenue Trend
daily_revenue = df.groupby("Order Date")["Revenue"].sum()
plt.figure(figsize=(12, 5))
plt.plot(daily_revenue.index, daily_revenue.values, marker="o", markersize=3, color="#1f77b4", linewidth=1.5)
plt.title("Daily Revenue Trend", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Order Date", fontsize=11)
plt.ylabel("Revenue ($)", fontsize=11)
plt.gca().yaxis.set_major_formatter('${x:,.0f}')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "daily_revenue_trend.png"), dpi=300)
plt.show()

# 7. Revenue Share by Channel
channel_revenue = df.groupby("Channel")["Revenue"].sum()
plt.figure(figsize=(7, 7))
plt.pie(channel_revenue.values, labels=channel_revenue.index, autopct="%1.1f%%", startangle=140, colors=["#1f77b4", "#ff7f0e", "#2ca02c"])
plt.title("Revenue Share by Channel", fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "revenue_share_channel.png"), dpi=300)
plt.show()

# 8. Revenue Share by Category (Donut)
category_revenue_pie = df.groupby("Category")["Revenue"].sum()
plt.figure(figsize=(7, 7))
plt.pie(category_revenue_pie.values, labels=category_revenue_pie.index, autopct="%1.1f%%", startangle=140, wedgeprops=dict(width=0.4, edgecolor='w'))
plt.title("Revenue Share by Category", fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "revenue_share_category.png"), dpi=300)
plt.show()

# 9. Revenue Distribution Histogram
plt.figure(figsize=(8, 5))
plt.hist(df["Revenue"], bins=15, color="#3949ab", edgecolor="black", alpha=0.8)
plt.title("Revenue Distribution", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Order Revenue ($)", fontsize=11)
plt.ylabel("Frequency", fontsize=11)
plt.gca().xaxis.set_major_formatter('${x:,.0f}')
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "revenue_distribution.png"), dpi=300)
plt.show()

# 10. Profit Distribution KDE
plt.figure(figsize=(8, 5))
sns.kdeplot(df["Profit"], fill=True, color="#00897b", linewidth=2)
plt.title("Profit Distribution", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Order Profit ($)", fontsize=11)
plt.gca().xaxis.set_major_formatter('${x:,.0f}')
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "profit_distribution.png"), dpi=300)
plt.show()

# 11. Profit Outlier Analysis Boxplot
plt.figure(figsize=(8, 5))
sns.boxplot(x=df["Profit"], color="#8e24aa")
plt.title("Profit Outlier Analysis", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Profit ($)", fontsize=11)
plt.gca().xaxis.set_major_formatter('${x:,.0f}')
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "profit_outliers.png"), dpi=300)
plt.show()

# 12. Revenue vs Profit Scatter Plot
plt.figure(figsize=(8, 5))
plt.scatter(df["Revenue"], df["Profit"], alpha=0.6, color="#d81b60", edgecolors="none")
plt.title("Revenue vs Profit", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Revenue ($)", fontsize=11)
plt.ylabel("Profit ($)", fontsize=11)
plt.gca().xaxis.set_major_formatter('${x:,.0f}')
plt.gca().yaxis.set_major_formatter('${x:,.0f}')
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "revenue_vs_profit.png"), dpi=300)
plt.show()

# 13. Revenue vs Profit Regression Plot
plt.figure(figsize=(8, 5))
sns.regplot(data=df, x="Revenue", y="Profit", scatter_kws={"alpha": 0.4}, line_kws={"color": "red", "linewidth": 2})
plt.title("Revenue vs Profit Relationship", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Revenue ($)", fontsize=11)
plt.ylabel("Profit ($)", fontsize=11)
plt.gca().xaxis.set_major_formatter('${x:,.0f}')
plt.gca().yaxis.set_major_formatter('${x:,.0f}')
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "revenue_vs_profit_regression.png"), dpi=300)
plt.show()

# 14. Number of Orders by Region
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="Region", palette="viridis")
plt.title("Number of Orders by Region", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Region", fontsize=11)
plt.ylabel("Number of Orders", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "orders_by_region.png"), dpi=300)
plt.show()

# 15. Number of Orders by Channel
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="Channel", palette="magma")
plt.title("Number of Orders by Channel", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Channel", fontsize=11)
plt.ylabel("Number of Orders", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "orders_by_channel.png"), dpi=300)
plt.show()

# 16. Revenue Heatmap by Region and Category
pivot_revenue = pd.pivot_table(df, values="Revenue", index="Region", columns="Category", aggfunc="sum")
plt.figure(figsize=(10, 6))
sns.heatmap(pivot_revenue, annot=True, fmt=",.0f", cmap="Blues", cbar_kws={"label": "Revenue ($)"})
plt.title("Revenue by Region and Category", fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "revenue_heatmap_region_category.png"), dpi=300)
plt.show()

# 17. Stacked Bar Chart - Revenue by Region & Category
pivot_revenue.plot(kind="bar", stacked=True, figsize=(10, 6), colormap="Blues")
plt.title("Revenue by Region and Category (Stacked)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Region", fontsize=11)
plt.ylabel("Revenue ($)", fontsize=11)
plt.xticks(rotation=0)
plt.gca().yaxis.set_major_formatter('${x:,.0f}')
plt.legend(title="Category", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "revenue_stacked_region_category.png"), dpi=300)
plt.show()

# 18. Revenue vs Profit by Region
region_data = df.groupby("Region")[["Revenue", "Profit"]].sum()
region_data.plot(kind="bar", figsize=(10, 6), color=["#1f77b4", "#2ca02c"])
plt.title("Revenue vs Profit by Region", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Region", fontsize=11)
plt.ylabel("Amount ($)", fontsize=11)
plt.xticks(rotation=0)
plt.gca().yaxis.set_major_formatter('${x:,.0f}')
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "revenue_vs_profit_by_region.png"), dpi=300)
plt.show()

# 19. Filled Area Plot - Revenue Trend
plt.figure(figsize=(12, 5))
plt.fill_between(daily_revenue.index, daily_revenue.values, color="#42a5f5", alpha=0.5)
plt.plot(daily_revenue.index, daily_revenue.values, color="#1e88e5", linewidth=1.5)
plt.title("Revenue Trend Area Plot", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Order Date", fontsize=11)
plt.ylabel("Revenue ($)", fontsize=11)
plt.gca().yaxis.set_major_formatter('${x:,.0f}')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "revenue_trend_area.png"), dpi=300)
plt.show()

# 20. Violin Plot - Profit Distribution by Category
plt.figure(figsize=(10, 6))
sns.violinplot(data=df, x="Category", y="Profit", palette="Set2")
plt.title("Profit Distribution by Category", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Category", fontsize=11)
plt.ylabel("Profit ($)", fontsize=11)
plt.gca().yaxis.set_major_formatter('${x:,.0f}')
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, "profit_by_category_violin.png"), dpi=300)
plt.show()

# 21. Pairplot Numeric Variables
pp = sns.pairplot(df[["Units Sold", "Unit Price", "Revenue", "Cost", "Profit", "Margin %"]], corner=True)
pp.fig.suptitle("Pairplot of Financial and Sales Metrics", y=1.02, fontsize=14, fontweight="bold")
plt.savefig(os.path.join(fig_dir, "pairplot_numeric.png"), dpi=300, bbox_inches="tight")
plt.show()

## 15. Key Business Insights

Based on the empirical analysis of 1,000 transactions across calendar year 2025, the following core business insights were calculated directly from the dataset:

1. **Overall Financial Performance**:
   - **Total Gross Revenue**: **$1,539,751.45**
   - **Total Operating Cost**: **$783,890.97**
   - **Total Net Profit**: **$755,860.48**
   - **Overall Profit Margin**: **49.09%** (Average transaction margin: **47.20%**)
   - **Total Units Sold**: **21,355 units** across 1,000 orders.

2. **Geographic Performance**:
   - **Top Region**: **South** led all regions in both Gross Revenue (**$434,395.49**) and Net Profit (**$218,807.00**), representing 28.2% of total revenue.
   - **Runner-up**: **North** followed with **$394,483.24** in revenue and **$191,773.03** in profit.
   - **East & West**: **East** (**$359,386.75**) and **West** (**$351,485.97**) performed closely in revenue, with profit margins remaining consistent around ~48-50% across all regions.

3. **Category & Product Highlights**:
   - **Revenue Leader**: **Home & Kitchen** generated the highest gross revenue (**$454,309.59**).
   - **Profitability Leader**: **Office** generated the highest net profit (**$207,714.30**).
   - **Top Product**: **Desk Chair** was the single most profitable product in the catalog, contributing **$139,953.60** in profit.
   - **Other High Performers**: **Cookware Set** (**$98,755.27** profit) and **Smart Watch** (**$90,627.81** profit).

4. **Sales Channel Breakdown**:
   - **Online Channel**: Dominates sales, accounting for **$802,328.49** (52.1% of total revenue) and **$391,374.02** in profit.
   - **Retail Store**: Second largest channel with **$482,622.42** (31.3% of revenue).
   - **Wholesale**: Generated **$254,800.54** (16.5% of revenue).

5. **Sales Representative Force**:
   - **Top Performer**: **K. Ivanova** delivered the highest individual revenue at **$243,152.03**.
   - **Top Group**: **T. Novak** (**$204,534.84**), **S. Owusu** (**$196,506.80**), and **A. Patel** (**$196,188.64**) were key top-tier contributors.

## 16. Conclusion & Recommendations

### Conclusion:
The sales dataset demonstrates strong financial health with an overall net profit margin of ~49%. Growth is heavily driven by the Online sales channel and the South regional market. Product-level analysis reveals concentrated profit drivers, particularly high-margin items in the Office and Home & Kitchen categories.

### Strategic Recommendations:
1. **Double Down on E-Commerce**: Expand digital marketing and online customer acquisition, given Online accounts for >52% of gross revenue.
2. **Scale South & North Expansion**: Reallocate inventory and logistics capabilities toward South and North regions where buyer demand and net margins are strongest.
3. **Optimize Product Portfolio**: Expand inventory and promotional focus on top-margin SKUs (e.g., Desk Chair, Cookware Set, Smart Watch) while reviewing margin structures for lower-tier items.
4. **Incentivize High Sales Performers**: Replicate sales strategies utilized by top representatives such as K. Ivanova and T. Novak across the broader sales force.